####**Project-Based Tile Installation Material Estimation & Recommendation System**

**Problem Statement**

The objective of this project is to develop an intelligent system that estimates tile installation material requirements (clips, wedges, tile grout) and recommends grout colours based on tile properties adn aesthetic preferences.

**Notebook Overview**
*   Generates a synthetic dataset using domain-based estimation and colour recommendation rules
*   Preprocesses the data by ensuring unit consistency and encoding categorical variables
*   Performs exploratory data analysis (EDA) to understand relationships between variables

**Libraries**

The following libraries are used for data generation, preprocessing, and exploratory data analysis.

In [ ]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
import math
import colorsys

**Input Ranges and Categories**

This section defines valid ranges and categories for input variables based on domain knowledge. These ranges are used to generate realistic synthetic data.

In [ ]:
# Installation area (in sq ft)
area_min = 100
area_max = 50000

# Tile dimensions (in inches)
tile_min = 4
tile_max = 48

# Tile thickness (in mm)
thickness_min = 6.4
thickness_max = 13.0

# Grout width (in mm)
grout_widths = [1.5, 2.0, 2.5, 3.0]

# Location categories
locations = ['floor', 'inside_wall', 'outside_wall']

# Tile color categories (for dataset generation)
tile_colors = [
    'white', 'ivory', 'beige', 'wood', 'chocolate', 'light_gray', 'dark_gray',
    'black'
]

# Aesthetic types
aesthetics = ['seamless', 'balanced', 'contrast']

**Helper Functions**

This section defines the helper functions required for dataset generation. These functions handle unit conversion, clip lookup, grout-related calculations, minimum grout width rules, and colour feature extraction.

In [ ]:
# 1. Unit conversion

def sqm_to_sqft(area_sqm):
    return area_sqm * 10.7639

def cm_to_inch(length_cm):
    return length_cm / 2.54

def sutar_to_mm(sutar):
    return sutar * 3.175  # 1 sutar = 1/8 inch = 3.175 mm

def inch_to_meter(x):
    return x * 0.0254

def mm_to_meter(x):
    return x / 1000

def sqft_to_sqm(area_sqft):
    return area_sqft * 0.092903


# 2. Unit standardization

def standardize_area(value, unit):
    if unit == "sqft":
        return value
    elif unit == "sqm":
        return sqm_to_sqft(value)
    else:
        raise ValueError("Unsupported area unit")

def standardize_dimension(value, unit):
    if unit == "inch":
        return value
    elif unit == "cm":
        return cm_to_inch(value)
    else:
        raise ValueError("Unsupported dimension unit")

def standardize_thickness(value, unit):
    if unit == "mm":
        return value
    elif unit == "sutar":
        return sutar_to_mm(value)
    else:
        raise ValueError("Unsupported thickness unit")


# 3. Clip lookup & quantity calculation

# Clips required per 100 sq ft
clip_lookup = {
    (4, 4): 1981,
    (6, 4): 1318, (6, 6): 877,
    (8, 4): 496, (8, 6): 329, (8, 8): 491,
    (10, 4): 398, (10, 6): 264, (10, 8): 390, (10, 10): 313,
    (12, 4): 660, (12, 6): 439, (12, 8): 489, (12, 10): 390, (12, 12): 430,
    (16, 4): 496, (16, 6): 329, (16, 8): 366, (16, 10): 292, (16, 12): 323, (16, 16): 241,
    (18, 4): 441, (18, 6): 293, (18, 8): 325, (18, 10): 259, (18, 12): 286, (18, 16): 214, (18, 18): 190,
    (20, 4): 397, (20, 6): 264, (20, 8): 292, (20, 10): 233, (20, 12): 257, (20, 16): 192, (20, 18): 170, (20, 20): 153,
    (22, 4): 361, (22, 6): 240, (22, 8): 265, (22, 10): 212, (22, 12): 233, (22, 16): 174, (22, 18): 155, (22, 20): 139, (22, 22): 126,
    (24, 4): 332, (24, 6): 220, (24, 8): 243, (24, 10): 194, (24, 12): 213, (24, 16): 159, (24, 18): 142, (24, 20): 127, (24, 22): 115, (24, 24): 106,
    (30, 4): 396, (30, 6): 264, (30, 8): 259, (30, 10): 207, (30, 12): 213, (30, 16): 160, (30, 18): 141, (30, 20): 128, (30, 22): 116, (30, 24): 105, (30, 30): 98,
    (32, 4): 372, (32, 6): 247, (32, 8): 242, (32, 10): 193, (32, 12): 199, (32, 16): 149, (32, 18): 132, (32, 20): 118, (32, 22): 108, (32, 24): 98, (32, 30): 94, (32, 32): 88,
    (35, 4): 328, (35, 6): 221, (35, 8): 215, (35, 10): 170, (35, 12): 177, (35, 16): 132, (35, 18): 117, (35, 20): 106, (35, 22): 95, (35, 24): 87, (35, 30): 83, (35, 32): 78, (35, 35): 69,
    (40, 4): 300, (40, 6): 198, (40, 8): 193, (40, 10): 154, (40, 12): 158, (40, 16): 117, (40, 18): 105, (40, 20): 94, (40, 22): 85, (40, 24): 78, (40, 30): 74, (40, 32): 69, (40, 35): 62, (40, 40): 55,
    (48, 4): 332, (48, 6): 220, (48, 8): 200, (48, 10): 161, (48, 12): 159, (48, 16): 118, (48, 18): 106, (48, 20): 95, (48, 22): 86, (48, 24): 79, (48, 30): 72, (48, 32): 67, (48, 35): 60, (48, 40): 53, (48, 48): 50
}

def get_clips_per_100sqft(tile_length_in, tile_width_in):
    key = tuple(sorted((tile_length_in, tile_width_in)))
    if key in clip_lookup:
        return clip_lookup[key]
    raise ValueError(f"Invalid tile size — must match dropdown options")

def calculate_clips(area_sqft, tile_length_in, tile_width_in):
    clips_per_100 = get_clips_per_100sqft(tile_length_in, tile_width_in)
    return math.ceil((area_sqft / 100) * clips_per_100)

def calculate_clip_boxes(clips, box_capacity=200):
    return math.ceil(clips / box_capacity)


# 4. Grout quantity calculation and minimum grout width logic

def calculate_grout_volume_ml(area_sqft, tile_length_in, tile_width_in, grout_width_mm, tile_thickness_mm, wastage_factor=1.15):
    area_sqm = sqft_to_sqm(area_sqft)
    tile_length_m = inch_to_meter(tile_length_in)
    tile_width_m = inch_to_meter(tile_width_in)
    grout_width_m = mm_to_meter(grout_width_mm)
    tile_thickness_m = mm_to_meter(tile_thickness_mm)

    grout_volume_m3 = (
        area_sqm
        * ((tile_length_m + tile_width_m) / (tile_length_m * tile_width_m))
        * grout_width_m
        * tile_thickness_m
        * wastage_factor
    )

    grout_volume_ml = grout_volume_m3 * 1_000_000
    return grout_volume_ml

def calculate_grout_tubes(grout_volume_ml, tube_size_ml=330):
    return math.ceil(grout_volume_ml / tube_size_ml)

def min_grout_width_by_tile(tile_length_in):
    if 4 <= tile_length_in <= 10:
        return 1.5
    elif 12 <= tile_length_in <= 22:
        return 2.0
    elif 24 <= tile_length_in <= 32:
        return 2.5
    elif 36 <= tile_length_in <= 48:
        return 3.0
    else:
        raise ValueError("Tile length outside supported range")

def min_grout_width_by_location(location):
    location_rules = {
        "floor": 2.0,
        "inside_wall": 1.5,
        "outside_wall": 2.5
    }
    return location_rules[location]

def get_min_required_grout_width(tile_length_in, location):
    return max(min_grout_width_by_tile(tile_length_in), min_grout_width_by_location(location))


# 5. Colour preprocessing

dropdown_to_hex = {
    "white": "#FFFFFF",
    "ivory": "#FFFFF0",
    "beige": "#F5F5DC",
    "light_gray": "#D3D3D3",
    "dark_gray": "#A9A9A9",
    "black": "#000000",
    "wood": "#8B5A2B",
    "chocolate": "#7B3F00"
}

def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

def normalize_tile_color(color_value, input_type="dropdown"):
    if input_type == "dropdown":
        return dropdown_to_hex[color_value]
    elif input_type == "hex":
        return color_value.upper()
    else:
        raise ValueError("Unsupported color input type")

def get_lightness(hex_color):
    r, g, b = hex_to_rgb(hex_color)
    brightness = (0.299 * r + 0.587 * g + 0.114 * b)
    if brightness < 85:
        return "dark"
    elif brightness < 170:
        return "medium"
    return "light"

def get_tone(hex_color):
    r, g, b = hex_to_rgb(hex_color)
    h, s, v = colorsys.rgb_to_hsv(r / 255, g / 255, b / 255)

    hue_deg = h * 360

    # True neutral (low saturation → greyish, no strong color)
    if s < 0.15:
        return "neutral"

    # Warm tones (reds, oranges, yellows)
    elif 0 <= hue_deg <= 70 or hue_deg >= 330:
        return "warm"

    # Cool tones (greens to blues)
    elif 160 <= hue_deg <= 260:
        return "cool"

    # Unclear hue (no strong warm/cool dominance)
    else:
        return "neutral"

# 6. Product mapping (primary recommendations)

available_colors = [
    {"name": "bright_white", "hex": "#FFFFFF", "tone": "neutral", "lightness": "light"},
    {"name": "glossy_black", "hex": "#000000", "tone": "neutral", "lightness": "dark"},
    {"name": "light_grey", "hex": "#D3D3D3", "tone": "cool", "lightness": "light"},
    {"name": "brown", "hex": "#8B4513", "tone": "warm", "lightness": "dark"},
    {"name": "beige", "hex": "#F5F5DC", "tone": "warm", "lightness": "light"},
    {"name": "ivory", "hex": "#FFFFF0", "tone": "warm", "lightness": "light"},
    {"name": "golden", "hex": "#DAA520", "tone": "warm", "lightness": "medium"},
]

def generate_primary_recommendations(tile_hex, lightness, tone):

    recommendations = {}

    # Seamless → closest OR no match

    seamless = None
    min_difference = float("inf")

    for color in available_colors:

        if color["tone"] == tone or color["tone"] == "neutral":

            color_val = 1 if color["lightness"] == "light" else 0.5 if color["lightness"] == "medium" else 0
            tile_val = 1 if lightness == "light" else 0.5 if lightness == "medium" else 0

            diff = abs(color_val - tile_val)

            if diff < min_difference:
                min_difference = diff
                seamless = color

    # if no strong match → mark missing
    if seamless is None or min_difference > 0.5:
        seamless = -1
    else:
        seamless = {
            "hex": seamless["hex"],
            "lightness": seamless["lightness"]
        }

    # Balanced → temperature anchors

    if tone == "cool":
        balanced = "light_grey"

    elif tone == "warm":
        balanced = random.choice(["beige", "ivory"])

    else:
        balanced = random.choice(["light_grey", "beige", "ivory"])

    # Contrast → strong definition

    if lightness == "light":
        contrast = random.choice(["glossy_black", "brown", "light_grey"])

    else:
        contrast = random.choice(["bright_white", "ivory"])

    # Golden rule

    if tone == "warm" and lightness == "dark":
        if random.random() < 0.3:
            contrast = "golden"

    return {
        "seamless": seamless,
        "balanced": balanced,
        "contrast": contrast
    }

# 7. Colour theory logic (additional recommendations)

warm_palette = [
    {"hex": "#D2B48C", "lightness": "medium"},  # tan
    {"hex": "#8B4513", "lightness": "dark"},    # brown
    {"hex": "#F5DEB3", "lightness": "light"}    # cream
]

cool_palette = [
    {"hex": "#D3D3D3", "lightness": "light"},   # light grey
    {"hex": "#FFFFFF", "lightness": "light"},   # white
    {"hex": "#36454F", "lightness": "dark"}     # charcoal
]

neutral_palette = [
    {"hex": "#FFFFFF", "lightness": "light"},   # white
    {"hex": "#D3D3D3", "lightness": "light"},   # light grey
    {"hex": "#000000", "lightness": "dark"}     # black
]

light_achromatic_pool = ["#FFFFFF", "#FFFFF0", "#F5F5DC"]   # white, ivory, cream
dark_achromatic_pool = ["#36454F", "#4F4F4F", "#000000"]    # charcoal, dark grey, black

def get_palette(tone):
    if tone == "warm":
        return warm_palette
    elif tone == "cool":
        return cool_palette
    else:
        return warm_palette + cool_palette + neutral_palette

def adjust_color(hex_color, lightness_factor=1.0, hue_shift=0):

    r, g, b = hex_to_rgb(hex_color)

    r, g, b = r/255, g/255, b/255

    h, l, s = colorsys.rgb_to_hls(r, g, b)

    # adjust lightness
    l = max(0, min(1, l * lightness_factor))

    # adjust hue (0–1 scale)
    h = (h + hue_shift) % 1.0

    r, g, b = colorsys.hls_to_rgb(h, l, s)

    return "#{:02X}{:02X}{:02X}".format(int(r*255), int(g*255), int(b*255))

def generate_additional_recommendations(tile_hex, lightness, tone):

    palette = get_palette(tone)

    # Seamless → monochromatic
    seamless = adjust_color(tile_hex, lightness_factor=1.02)

    # Balanced → analogous OR neutral
    analogous = adjust_color(tile_hex, hue_shift=0.08)  # ~30°
    neutral = random.choice(palette)["hex"]
    balanced = random.choice([analogous, neutral])

    # Contrast → complementary OR achromatic extreme
    complementary = adjust_color(tile_hex, hue_shift=0.5)

    if lightness == "light":
        achromatic = random.choice(dark_achromatic_pool)
    else:
        achromatic = random.choice(light_achromatic_pool)

    contrast = random.choice([complementary, achromatic])

    return {
        "seamless": seamless,
        "balanced": balanced,
        "contrast": contrast
    }


**Synthetic Dataset Generation**

This section generates structured, diverse data by systematically varying tile properties and applying estimation and recommendation logic.

In [ ]:
data = []

# define ranges (controlled diversity)
areas = [100, 500, 1000, 5000, 10000, 20000]
tile_sizes = [4, 6, 8, 10, 12, 16, 24, 32, 48]
locations = ["floor", "inside_wall", "outside_wall"]

tile_colors = list(dropdown_to_hex.keys())

n_samples = 200

for _ in range(n_samples):

    # 1. Generate inputs

    area = random.choice(areas)

    tile_length = random.choice(tile_sizes)
    tile_width = random.choice(tile_sizes)

    tile_thickness = round(random.uniform(6.4, 13), 2)

    location = random.choice(locations)

    # mix dropdown + random hex
    if random.random() < 0.7:
        tile_color_input = random.choice(tile_colors)
        tile_hex = normalize_tile_color(tile_color_input, "dropdown")
    else:
        tile_hex = "#{:06X}".format(random.randint(0, 0xFFFFFF))

    # 2. Extract properties

    lightness = get_lightness(tile_hex)
    tone = get_tone(tile_hex)

    # 3. Quantity calculations

    clips = calculate_clips(area, tile_length, tile_width)

    grout_width = get_min_required_grout_width(tile_length, location)

    grout_volume = calculate_grout_volume_ml(
        area, tile_length, tile_width, grout_width, tile_thickness
    )

    grout_tubes = calculate_grout_tubes(grout_volume)

    # 4. Recommendations

    primary = generate_primary_recommendations(tile_hex, lightness, tone)
    additional = generate_additional_recommendations(tile_hex, lightness, tone)

    # 5. Store row

    data.append({
        "area_sqft": area,
        "tile_length_in": tile_length,
        "tile_width_in": tile_width,
        "tile_thickness_mm": tile_thickness,
        "location": location,

        "tile_hex": tile_hex,
        "lightness": lightness,
        "tone": tone,

        "clips": clips,
        "grout_volume_ml": grout_volume,
        "grout_tubes": grout_tubes,

        "primary": primary,
        "additional": additional
    })

**Data Preprocessing**

This section covers the raw dataset into a structured format by flattening recommendation fields and encoding categorical variables for machine learning.

In [ ]:
# 1. Convert to dataframe

df = pd.DataFrame(data)


# 2. Flatten recommendation columns

df["seamless_primary"] = df["primary"].apply(lambda x: x["seamless"])
df["balanced_primary"] = df["primary"].apply(lambda x: x["balanced"])
df["contrast_primary"] = df["primary"].apply(lambda x: x["contrast"])

df["seamless_additional"] = df["additional"].apply(lambda x: x["seamless"])
df["balanced_additional"] = df["additional"].apply(lambda x: x["balanced"])
df["contrast_additional"] = df["additional"].apply(lambda x: x["contrast"])

# drop nested columns
df = df.drop(columns=["primary", "additional"])


# 3. Handle seamless = None → -1

df["seamless_primary"] = df["seamless_primary"].apply(lambda x: -1 if x == -1 else x)


# 4. Encode categorical inputs

# One-hot encoding for location, tone, lightness
df = pd.get_dummies(df, columns=["location", "tone", "lightness"])


# 5. Encode output labels

color_to_label = {
    "bright_white": 0,
    "glossy_black": 1,
    "light_grey": 2,
    "brown": 3,
    "beige": 4,
    "ivory": 5,
    "golden": 6
}

def encode_color(val):
    if val == -1:
        return -1
    return color_to_label.get(val, -1)

# apply encoding
for col in [
    "seamless_primary", "balanced_primary", "contrast_primary",
    "seamless_additional", "balanced_additional", "contrast_additional"
]:
    df[col] = df[col].apply(encode_color)


# 6. Final check

df.head()

**Exploratory Data Analysis (EDA)**

This section analyzes the dataset to check distributions, detect imbalance, and ensure data quality before model training.

In [ ]:
# 1. Basic info

print(df.shape)
df.info()


# 2. Summary statistics

df.describe()


# 3. Check missing values

df.isnull().sum()


# 4. Distribution of key variables

import matplotlib.pyplot as plt

# area distribution
plt.hist(df["area_sqft"], bins=20)
plt.title("Area Distribution")
plt.xlabel("Area (sqft)")
plt.ylabel("Frequency")
plt.show()

# tile size distribution
plt.hist(df["tile_length_in"], bins=15)
plt.title("Tile Length Distribution")
plt.show()


# 5. Categorical balance check

# tone
df.filter(like="tone_").sum()

# lightness
df.filter(like="lightness_").sum()

# location
df.filter(like="location_").sum()


# 6. Output distribution

df["balanced_primary"].value_counts()
df["contrast_primary"].value_counts()


# 7. Check -1 occurrences

(df["seamless_primary"] == -1).sum()